# SFTF Exploratory Analyses & Secondary Tuning Metrics

This notebook contains the exploratory and supplementary analyses of drifting grating recordings, including grating trial metrics, parametric tuning fits, direction/orientation selectivity, and speed-dependent shift model comparisons. Moved from .

In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
from functools import partial
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from PIL import Image
import flexiznam as flz

from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM
from cottage_analysis.analysis import common_utils
from cottage_analysis.preprocessing import synchronisation
from cottage_analysis.analysis.fit_gaussian_blob import GratingParams, grating_tuning
from v1_depth_map.figure_utils.rf_distance import load_session_neurons
from v1_depth_map.paths import get_figures_roots

style.setup_figure_fonts()

PROJECT = "hey2_3d-vision_foodres_20220101"
SESSIONS = [
    "PZAH10.2d_S20230920",
    "PZAH10.2d_S20230922",
    "PZAH10.2f_S20230817",
    "PZAH10.2f_S20230912",
    "PZAH10.2f_S20230914",
    "PZAH10.2f_S20230924",
]
PHOTODIODE_PROTOCOL = 5  # all PZAH mice
TRACES = {"anatomical_only": 3, "ast_neuropil": False}
PARAMS = ["SpatialFrequency", "TemporalFrequency", "Angle"]
LAG = 8  # onset volumes skipped (0.53 s at 15.2 Hz; window ends at the next onset, as gratings.generate_trials_df)
FRAME_LAG = 0  # frames between stimulus / RS and dF/F in the frame-level analyses (as the RS/OF fits)
MIN_VOLUMES = 10  # drop trials shorter than this (truncated at the end of a recording)
P_RESPONSIVE = 0.01  # ANOVA across conditions
P_SELECTIVE = 0.05  # gOSI / gDSI shuffle
N_SHUFFLES = 1000
REBUILD = False  # True: re-sync the recordings instead of reading the trial cache
# selection criteria and stats of the thesis SFTF figure (yiran-phd-thesis/yh_thesis/depth/SFTF.ipynb),
# except SF/TF tuned, which is cross-validated (one-sided Spearman p < 0.05, as for depth)
# RS/OF tuned (panels B, C): rsof_test_rsq_closedloop_g2d above the empirical-null threshold of figure 2
# (mirrored negative tail, common_utils.add_rsq_significance)
RSOF_SIG_PERCENTILE = 95
DSI_THR = 0.6  # panel F: direction-selective cells only
N_BOOTS = 20000  # hierarchical bootstrap (mouse, session) of the B, C correlations (exported stats)
N_BOOTS_EXPLORATORY = 2000  # exploratory sections (not exported): p resolution 5e-4

GROUPS = {"depth tuned": "#2a78d6", "not depth tuned": "#eb6834"}

flm = flz.get_flexilims_session(PROJECT)
_, SAVE_ROOT = get_figures_roots(flm, fig_subdir="fig_reviewer_gratings")
CACHE = SAVE_ROOT / "cache"
FRAMES_DIR = SAVE_ROOT / "running_models" / "frames"  # frame-level data of the running-speed models
CACHE.mkdir(parents=True, exist_ok=True)
print(f"Saving to {SAVE_ROOT}")

In [ ]:
# Load data and precomputed statistics for figure
csv_path = SAVE_ROOT / "grating_tuning_cells.csv"
cells = pd.read_csv(csv_path)

depth_sftf = cells[cells.depth_tuned & cells.sftf_tuned].reset_index(drop=True)

tuned = cells[cells.sftf_tuned & cells.depth_tuned]
tuned_other = cells[cells.sftf_tuned & ~cells.depth_tuned]

DSI_THR = 0.6
dir_tuned = tuned[tuned.fit_dsi > DSI_THR]
dir_tuned_other = tuned_other[tuned_other.fit_dsi > DSI_THR]

# Load example cell raw trials from cache (PZAH10.2d_S20230922 ROI 558: preferred depth 0.70 m)
example_session = "PZAH10.2d_S20230922"
example_roi = 558
trials_cache = CACHE / f"{example_session}_sftf_trials.pickle"
trials_data = pd.read_pickle(trials_cache)

trials_df = trials_data["trials"].copy()
trials_df["response"] = trials_data["dff"][:, example_roi]

angle_range = np.sort(trials_df.Angle.unique())
sf_range = np.sort(trials_df.SpatialFrequency.unique())
tf_range = np.sort(trials_df.TemporalFrequency.unique())

all_angles = np.zeros((len(angle_range), len(sf_range), len(tf_range)))
for i, angle in enumerate(angle_range):
    this_df = trials_df[trials_df.Angle == angle]
    for isf, sf in enumerate(sf_range):
        for itf, tf in enumerate(tf_range):
            all_angles[i, isf, itf] = this_df[
                (this_df.SpatialFrequency == sf) & (this_df.TemporalFrequency == tf)
            ]["response"].mean()

vmax = np.round(np.nanmax(all_angles), 1)

# Load example cell depth tuning data
depth_example_cache = CACHE / f"{example_session}_roi{example_roi}_depth_example.pickle"
if depth_example_cache.exists():
    depth_example_data = pd.read_pickle(depth_example_cache)
    example_depth_trials = depth_example_data["trials_depth"]
    example_depth_neurons = depth_example_data["neurons_df"].copy()
    example_depth_neurons.index = [0]
else:
    from v1_depth_map.figure_utils import sftf

    example_depth_trials = sftf.load_depth_trials(flm, example_session)
    example_depth_neurons, _ = load_session_neurons(flm, example_session)

import contextlib, io


def bootstrap_spearman(x, y, sessions, n_boots=N_BOOTS):
    # Hierarchical bootstrap resampled on session then cells
    df_boot = pd.DataFrame(
        dict(x=np.asarray(x), y=np.asarray(y), session=np.asarray(sessions))
    )
    df_boot = df_boot[np.isfinite(df_boot.x) & np.isfinite(df_boot.y)].reset_index(
        drop=True
    )
    with contextlib.redirect_stderr(io.StringIO()):
        rho, dist = common_utils.hierarchical_bootstrap_stats(
            df_boot,
            n_boots,
            ["x"],
            ["session"],
            ycol=["y"],
            correlation=True,
            plot=False,
        )
    return dict(
        n=len(df_boot),
        rho=float(rho[0]),
        ci=np.percentile(dist[:, 0], [2.5, 97.5]),
        p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
    )


# Precompute / load bootstrap statistics for F, G, H
stat_sf_depth = bootstrap_spearman(
    np.exp(depth_sftf.fit_sf0), depth_sftf.preferred_depth * 100, depth_sftf.session
)
stat_tf_depth = bootstrap_spearman(
    np.exp(depth_sftf.fit_tf0), depth_sftf.preferred_depth * 100, depth_sftf.session
)
stat_speed_depth = bootstrap_spearman(
    depth_sftf.pref_speed, depth_sftf.preferred_depth * 100, depth_sftf.session
)

print(
    f"SF vs depth: n = {stat_sf_depth['n']}, rho = {stat_sf_depth['rho']:.3f}, 95% CI {stat_sf_depth['ci'].round(3)}, p = {stat_sf_depth['p']:.4f}"
)
print(
    f"TF vs depth: n = {stat_tf_depth['n']}, rho = {stat_tf_depth['rho']:.3f}, 95% CI {stat_tf_depth['ci'].round(3)}, p = {stat_tf_depth['p']:.4f}"
)
print(
    f"Speed vs depth: n = {stat_speed_depth['n']}, rho = {stat_speed_depth['rho']:.3f}, 95% CI {stat_speed_depth['ci'].round(3)}, p = {stat_speed_depth['p']:.4f}"
)

# Cells tuned to both SFTF and RS-OF (g2d) for Panel I (OF vs SF)
both = cells[cells.sftf_tuned & cells.rsof_tuned].reset_index(drop=True)
stat_sf_of = bootstrap_spearman(
    np.exp(both.fit_sf0), both.preferred_of, both.session
)
print(
    f"SF vs OF: n = {stat_sf_of['n']}, rho = {stat_sf_of['rho']:.3f}, 95% CI {stat_sf_of['ci'].round(3)}, p = {stat_sf_of['p']:.4f}"
)

# Load held-out population data and example cell configurations for Panels I-L
summary_cache = CACHE / "fig3_sftf_summary_n20000.pickle"
summary_data = pd.read_pickle(summary_cache)
heldout_means = summary_data["heldout_means"]
mean_peaks = summary_data["mean_peaks"]
speed_labels = summary_data["speed_labels"]

SPEED_EDGES_CM_S = [5, 11.31, 22.63, 45.25, 90.51]  # first bin from 5 cm/s, the still threshold (STILL_THR)
TF_SELECT_THRESHOLD = 0.5
STILL_DIR = SAVE_ROOT / "running_models" / "frames_still"

# 2 exemplar depth-selective cells showing clear rightward shifts in preferred TF with locomotion
EXAMPLE_CELLS_SPEED = [
    ("PZAH10.2f_S20230924", 95, "Cell 1"),
    ("PZAH10.2d_S20230922", 562, "Cell 2"),
]
# Load static population data for Panels K & L
static_pop_cache = CACHE / 'fig3_sftf_static_pop.pickle'
if static_pop_cache.exists():
    static_pop_data = pd.read_pickle(static_pop_cache)
    static_means = static_pop_data['means']
    static_peaks = static_pop_data['peaks']
else:
    static_means = None
    static_peaks = {}


---
# Detailed analyses, exploratory sections, and statistical tests

The sections below contain the original analysis pipeline, model-free comparisons, running speed model evaluations, and exported statistical summaries.

## Grating trials

One row per trial, response = mean dF/F of each ROI over the trial window. Trials are split where the
parameter-log time (`stimulus_harptime`) changes, so two consecutive trials with the same parameters stay
separate. The last trial of a recording gets the median trial length. The trial tables are cached
per session.

In [ ]:
def recording_trials(recording):
    # Trial table and (n_trials, n_rois) mean dF/F of one SFTF recording.
    vs_df = synchronisation.generate_vs_df(
        recording=recording,
        photodiode_protocol=PHOTODIODE_PROTOCOL,
        flexilims_session=flm,
        project=PROJECT,
    )
    img_df = synchronisation.generate_imaging_df(
        vs_df=vs_df,
        recording=recording,
        flexilims_session=flm,
        filter_datasets=TRACES,
        return_volumes=True,
    ).reset_index(drop=True)
    dff = np.vstack(img_df.dffs.values)  # (n_volumes, n_rois)

    stim = img_df["stimulus_harptime"]
    onset = (stim.notna() & (stim != stim.shift())).values
    starts = np.flatnonzero(onset)
    ends = np.append(starts[1:], starts[-1] + int(np.median(np.diff(starts))))
    starts, ends = starts + LAG, np.minimum(ends, len(img_df))
    keep = ends - starts >= MIN_VOLUMES

    trials = img_df.loc[onset, PARAMS].reset_index(drop=True)
    trials["n_volumes"] = ends - starts
    trials["recording"] = recording["name"]
    resp = np.stack([dff[s:e].mean(axis=0) for s, e in zip(starts, ends)])
    return trials[keep].reset_index(drop=True), resp[keep]


def session_trials(session):
    # All SFTF trials of a session, cached in CACHE.
    path = CACHE / f"{session}_sftf_trials.pickle"
    if path.exists() and not REBUILD:
        return pd.read_pickle(path)
    recordings = flz.get_children(
        parent_name=session, children_datatype="recording", flexilims_session=flm
    )  # indexed by name, which the sync functions read as `recording.name`
    recordings = recordings[recordings["name"].str.contains("SFTF")].sort_index()
    out = [recording_trials(rec) for _, rec in recordings.iterrows()]
    data = dict(
        trials=pd.concat([o[0] for o in out], ignore_index=True),
        dff=np.concatenate([o[1] for o in out], axis=0),
    )
    pd.to_pickle(data, path)
    return data


data = {session: session_trials(session) for session in SESSIONS}

In [ ]:
summary = []
for session, d in data.items():
    t = d["trials"]
    reps = t.groupby(PARAMS).size()
    summary.append(
        dict(
            session=session,
            recordings=t.recording.nunique(),
            trials=len(t),
            rois=d["dff"].shape[1],
            conditions=len(reps),
            reps_min=reps.min(),
            reps_max=reps.max(),
            volumes_per_trial=t.n_volumes.median(),
        )
    )
summary = pd.DataFrame(summary)
summary

## Tuning metrics

In [ ]:
def vector_metrics(curves, theta):
    # gOSI, gDSI, preferred orientation and direction (deg) of rectified curves (..., n_dir).
    r = np.clip(curves, 0, None)
    total = r.sum(axis=-1)
    v1 = (r * np.exp(1j * theta)).sum(axis=-1)
    v2 = (r * np.exp(2j * theta)).sum(axis=-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        gosi, gdsi = np.abs(v2) / total, np.abs(v1) / total
    pref_ori = np.rad2deg(np.angle(v2)) / 2 % 180
    pref_dir = np.rad2deg(np.angle(v1)) % 360
    return gosi, gdsi, pref_ori, pref_dir


def session_tuning(trials, resp, rng):
    values = [np.unique(trials[p]) for p in PARAMS]
    n_sf, n_tf, n_dir = (len(v) for v in values)
    i_sf, i_tf, i_dir = (np.searchsorted(v, trials[p]) for v, p in zip(values, PARAMS))
    i_sftf = i_sf * n_tf + i_tf
    cond = i_sftf * n_dir + i_dir
    n_cond, n_roi = n_sf * n_tf * n_dir, resp.shape[1]
    theta = np.deg2rad(values[2])

    counts = np.bincount(cond, minlength=n_cond)
    sums = np.zeros((n_cond, n_roi))
    np.add.at(sums, cond, resp)
    with np.errstate(invalid="ignore"):
        means = (sums / counts[:, None]).reshape(n_sf * n_tf, n_dir, n_roi)
    best = np.nanargmax(np.nanmean(means, axis=1), axis=0)  # preferred SF/TF per ROI
    curves = means[best, :, np.arange(n_roi)]  # (n_roi, n_dir)
    gosi, gdsi, pref_ori, pref_dir = vector_metrics(curves, theta)

    # shuffle direction labels among the trials of each ROI's preferred SF/TF
    p_osi, p_dsi = np.full(n_roi, np.nan), np.full(n_roi, np.nan)
    for k in np.unique(best):
        rois = np.flatnonzero(best == k)
        tr = np.flatnonzero(i_sftf == k)
        labels = np.tile(i_dir[tr], (N_SHUFFLES, 1))
        onehot = np.eye(n_dir)[
            rng.permuted(labels, axis=1)
        ]  # (n_shuffles, n_trials, n_dir)
        onehot /= np.bincount(i_dir[tr], minlength=n_dir)
        shuffled = np.einsum("std,tr->srd", onehot, resp[np.ix_(tr, rois)])
        osi_sh, dsi_sh, _, _ = vector_metrics(shuffled, theta)
        p_osi[rois] = (1 + (osi_sh >= gosi[rois]).sum(axis=0)) / (N_SHUFFLES + 1)
        p_dsi[rois] = (1 + (dsi_sh >= gdsi[rois]).sum(axis=0)) / (N_SHUFFLES + 1)

    p_resp = stats.f_oneway(*[resp[cond == c] for c in range(n_cond)], axis=0).pvalue
    return (
        pd.DataFrame(
            dict(
                roi=np.arange(n_roi),
                pref_sf=values[0][best // n_tf],
                pref_tf=values[1][best % n_tf],
                gosi=gosi,
                gdsi=gdsi,
                pref_ori=pref_ori,
                pref_dir=pref_dir,
                p_osi=p_osi,
                p_dsi=p_dsi,
                p_resp=p_resp,
            )
        ),
        curves,
    )


NEURONS_COLUMNS = [
    "roi",
    "iscell",
    "depth_tuned",
    "preferred_depth_closedloop",
    "preferred_OF_closedloop_g2d",
    "preferred_RS_closedloop_g2d",
    "rsof_test_rsq_closedloop_g2d",
    "rsof_popt_closedloop_g2d",
]


def session_neurons(session):
    # columns of load_session_neurons used here, cached (the full neurons_df is ~150-325 MB)
    path = CACHE / f"{session}_neurons_slim.pickle"
    if path.exists() and not REBUILD:
        return pd.read_pickle(path)
    neurons_df, _ = load_session_neurons(flm, session)
    slim = neurons_df[NEURONS_COLUMNS].reset_index(drop=True)
    slim.to_pickle(path)
    return slim


rng = np.random.default_rng(0)
cells, curves = [], {}
for session in SESSIONS:
    neurons_df = session_neurons(session)
    tuning, curves[session] = session_tuning(
        data[session]["trials"], data[session]["dff"], rng
    )
    assert len(tuning) == len(
        neurons_df
    ), f"{session}: ROI count differs from neurons_df"
    tuning["session"] = session
    tuning["iscell"] = neurons_df.iscell.values.astype(bool)
    tuning["depth_tuned"] = neurons_df.depth_tuned.values.astype(bool)
    tuning["preferred_depth"] = neurons_df.preferred_depth_closedloop.values
    # RS/OF Gaussian fit of the depth protocol, rad/s -> deg/s
    tuning["preferred_of"] = np.degrees(
        neurons_df.preferred_OF_closedloop_g2d.astype(float).values
    )
    tuning["rsof_test_rsq"] = neurons_df.rsof_test_rsq_closedloop_g2d.astype(
        float
    ).values
    tuning["preferred_rs"] = neurons_df.preferred_RS_closedloop_g2d.astype(
        float
    ).values  # m/s
    cells.append(tuning)
cells = pd.concat(cells, ignore_index=True)
cells = cells[cells.iscell].reset_index(drop=True)
cells["group"] = np.where(cells.depth_tuned, "depth tuned", "not depth tuned")
cells["responsive"] = cells.p_resp < P_RESPONSIVE
cells["ori_selective"] = cells.responsive & (cells.p_osi < P_SELECTIVE)
cells["dir_selective"] = cells.responsive & (cells.p_dsi < P_SELECTIVE)
print(f"{len(cells)} cells")

## Parametric fit

`neurons_df_sftf.pickle` of each session (see `revisions/sftf/NEURONS_DF_SFTF.md`): `fit_sftf_tuning` on
single-trial responses, a 2D Gaussian in log SF × log TF times a von Mises direction tuning with a direction
selectivity parameter `dsi` (0: equal response to both directions of an orientation, 1: no response to the
opposite direction). Preferred speed = TF0 / SF0 (deg/s), preferred direction = `alpha0`, preferred
orientation = `alpha0` mod 180°. Each ROI is also fitted on 4 of the 5 recordings and evaluated on the
5th; a cell is *SF/TF tuned* if the Spearman correlation between held-out responses and predictions is
significantly positive (one-sided p < 0.05, `add_one_sided_spearman_significance`, as for depth tuning).

In [ ]:
from cottage_analysis.analysis import common_utils
from cottage_analysis.pipelines import pipeline_utils


def load_sftf_fit(session):
    folder = pipeline_utils.create_neurons_ds(
        session_name=session, flexilims_session=flm, project=PROJECT, conflicts="skip"
    ).path_full.parent
    fit = pd.read_pickle(folder / "neurons_df_sftf.pickle")
    assert (
        "sftf_test_spearmanr_pval" in fit
    ), f"{session}: neurons_df_sftf has no cross-validation"
    fit.insert(0, "roi", np.arange(len(fit)))
    fit.insert(0, "session", session)
    return fit.drop(columns=["sftf_test_popts"])


fits = pd.concat([load_sftf_fit(s) for s in SESSIONS], ignore_index=True)
cells = cells.drop(columns=[c for c in cells.columns if c.startswith("fit_")]).merge(
    fits.rename(
        columns={c: f"fit_{c}" for c in fits.columns if c not in ("session", "roi")}
    ),
    on=["session", "roi"],
    how="left",
    validate="one_to_one",
)
cells["pref_speed"] = np.exp(cells.fit_tf0 - cells.fit_sf0)  # deg/s
cells["fit_dir"] = np.rad2deg(cells.fit_alpha0) % 360
cells["fit_ori"] = cells.fit_dir % 180
common_utils.add_one_sided_spearman_significance(
    cells,
    rval_col="fit_sftf_test_spearmanr_rval",
    pval_col="fit_sftf_test_spearmanr_pval",
    out_col="sftf_tuned",
)
rsof_thresholds = common_utils.add_rsq_significance(
    cells, {"g2d": "rsof_test_rsq"}, percentile=RSOF_SIG_PERCENTILE, verbose=False
)
cells["rsof_tuned"] = cells["rsof_test_rsq_sig"]
cells.to_csv(SAVE_ROOT / "grating_tuning_cells.csv", index=False)
print(f"SF/TF tuned (cross-validated): {cells.sftf_tuned.sum()} / {len(cells)}")
print(f"  for reference, in-sample R² > 0.05 (thesis): {(cells.fit_rsq > 0.05).sum()}")
print(
    f"RS/OF tuned (test R² > empirical-null threshold {rsof_thresholds}): {cells.rsof_tuned.sum()}"
)
print(
    f"SF/TF tuned and depth selective: {(cells.sftf_tuned & cells.depth_tuned).sum()}"
)
print(f"SF/TF tuned and RS/OF tuned: {(cells.sftf_tuned & cells.rsof_tuned).sum()}")

In [ ]:
# in-sample vs held-out fit quality
fig, axes = plt.subplots(1, 2, figsize=(12 * CM, 5 * CM))
ax = axes[0]
ax.scatter(
    cells.fit_rsq, cells.fit_sftf_test_rsq, s=3, color="k", alpha=0.3, linewidths=0
)
lim = [min(cells.fit_sftf_test_rsq.min(), 0), cells.fit_rsq.max()]
ax.plot(lim, lim, color="#8a8984", lw=0.8, ls="--")
ax.set_xlabel("In-sample R²")
ax.set_ylabel("Held-out R²")
ax = axes[1]
ax.hist(cells.fit_sftf_test_spearmanr_rval, bins=50, color="k")
ax.set_xlabel("Held-out Spearman r")
ax.set_ylabel("Number of cells")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "sftf_crossval.pdf", fig=fig, bbox_inches="tight")

## Summary figure

- **A**: overlap of depth-selective and SF/TF-tuned cells (areas proportional to cell counts).
- **B**: preferred grating speed vs preferred optic flow speed (depth protocol RS/OF fit), for cells that are SF/TF
  tuned and RS/OF tuned (RS/OF test R² above the empirical-null threshold, as in figure 2; fits on a bound are kept).
- **C**: preferred grating speed vs preferred virtual depth, for cells that are SF/TF tuned and depth selective
  (preferred depth is only meaningful for depth-selective cells; the thesis used the B selection).
  B and C: Spearman ρ with a hierarchical bootstrap over mice and sessions (`N_BOOTS`), as in the thesis.
  Dashed line in B: identity.
- **D, E**: direction selectivity (`dsi`) and preferred orientation of SF/TF-tuned cells: depth-selective (black
  outline) and not depth-selective (light grey, filled); proportion of cells in each group.
- **F**: preferred direction of the same cells with `dsi` > `DSI_THR`.

In [ ]:
from scipy.optimize import brentq


def circle_overlap(r1, r2, d):
    # Area of the intersection of two circles of radii r1, r2 with centres d apart.
    if d >= r1 + r2:
        return 0.0
    if d <= abs(r1 - r2):
        return np.pi * min(r1, r2) ** 2
    a1 = r1**2 * np.arccos((d**2 + r1**2 - r2**2) / (2 * d * r1))
    a2 = r2**2 * np.arccos((d**2 + r2**2 - r1**2) / (2 * d * r2))
    return (
        a1
        + a2
        - 0.5 * np.sqrt((-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2))
    )


def plot_venn(ax, n_all, n_a, n_b, n_ab, names, colors):
    # Area-proportional Venn of sets a and b inside the set of all cells.
    R, ra, rb = np.sqrt(np.array([n_all, n_a, n_b]) / np.pi)
    lo, hi = abs(ra - rb), ra + rb
    d = brentq(lambda x: circle_overlap(ra, rb, x) - n_ab, lo + 1e-9, hi - 1e-9)
    xa = -(ra + d + rb) / 2 + ra
    xb = xa + d
    # highest position keeping both circles inside the outer one
    y = 0.97 * min(
        np.sqrt(max((R - r) ** 2 - x**2, 0)) for r, x in [(ra, xa), (rb, xb)]
    )
    ax.add_patch(plt.Circle((0, 0), R, color="#d9d8d4", lw=0))
    ax.add_patch(plt.Circle((xa, y), ra, color=colors[0], alpha=0.6, lw=0))
    ax.add_patch(plt.Circle((xb, y), rb, color=colors[1], alpha=0.6, lw=0))
    pct = lambda n: f"{100 * n / n_all:.0f}%"
    for x, n in [
        ((xa - ra + xb - rb) / 2, n_a - n_ab),
        ((xb - rb + xa + ra) / 2, n_ab),
        ((xa + ra + xb + rb) / 2, n_b - n_ab),
    ]:
        ax.text(x, y, pct(n), ha="center", va="center")
    ax.text(0, -0.75 * R, pct(n_all - n_a - n_b + n_ab), ha="center", va="center")
    ax.text(xa - ra, y + ra, names[0], ha="right", va="bottom")
    ax.text(xb + 0.3 * rb, y + rb, names[1], ha="left", va="bottom")
    ax.text(0, -R * 1.02, f"All neurons (n={n_all})", ha="center", va="top")
    ax.set_xlim(-R * 1.05, R * 1.05)
    ax.set_ylim(-R * 1.25, R * 1.15)
    ax.set_aspect("equal")
    ax.axis("off")


def polar_hist(
    ax, values_deg, period, n_bins, step=False, color="k", proportion=True, lw=1
):
    # Histogram on the polar axes, bins centred on multiples of period / n_bins (the stimulus directions when
    # n_bins = 8), as filled bars or (step=True) an unfilled outline; proportion of cells by default.
    width = period / n_bins
    counts, _ = np.histogram(
        (values_deg + width / 2) % period, bins=n_bins, range=(0, period)
    )
    heights = counts / counts.sum() if proportion else counts
    if step:
        theta = np.linspace(0, period, 721)
        r = heights[((theta + width / 2) // width).astype(int) % n_bins]
        ax.plot(np.deg2rad(theta), r, color=color, lw=lw, zorder=3)
    else:
        centres = np.deg2rad(np.arange(n_bins) * width)
        ax.bar(centres, heights, width=np.deg2rad(width) * 0.95, color=color, lw=0)
        if period < 360:  # the first bin wraps around: draw its other half at `period`
            ax.bar(
                np.deg2rad(period),
                heights[0],
                width=np.deg2rad(width) * 0.95,
                color=color,
                lw=0,
            )
    ax.set_thetamin(0)
    ax.set_thetamax(period)
    ax.set_xticks(np.deg2rad(np.arange(0, period + 1 if period < 360 else period, 45)))
    ax.set_rlabel_position(
        period / n_bins / 2
    )  # radial labels between the first two bars
    ax.yaxis.set_major_locator(plt.MaxNLocator(3))
    ax.tick_params(labelsize=5)
    return heights


def log_scatter(ax, x, y, stat, xlabel, ylabel):
    ax.scatter(x, y, s=8, color="k", alpha=0.6, linewidths=0)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ci = stat["ci"]
    ax.set_title(
        f"n = {stat['n']}, ρ = {stat['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}], bootstrap p = {stat['p']:.2g}",
        fontsize=6,
    )

In [ ]:
# summary figure statistics: cell selections, Venn counts, B and C correlations
import contextlib
import io

from cottage_analysis.analysis import common_utils


def bootstrap_spearman(x, y, sessions, n_boots=N_BOOTS):
    # Spearman rho, 95% CI and p from a hierarchical bootstrap over mice then sessions (thesis stats)
    df = pd.DataFrame(
        dict(x=np.asarray(x), y=np.asarray(y), session=np.asarray(sessions))
    )
    df = df[np.isfinite(df.x) & np.isfinite(df.y)].reset_index(drop=True)
    with contextlib.redirect_stderr(io.StringIO()):  # silence the tqdm progress bar
        rho, dist = common_utils.hierarchical_bootstrap_stats(
            df,
            n_boots,
            ["x"],
            ["mouse", "session"],
            ycol=["y"],
            correlation=True,
            plot=False,
        )
    return dict(
        n=len(df),
        n_sessions=df.session.nunique(),
        rho=float(rho[0]),
        ci=np.percentile(dist[:, 0], [2.5, 97.5]),
        p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
    )


tuned = cells[cells.sftf_tuned & cells.depth_tuned]  # D, E
dir_tuned = tuned[tuned.fit_dsi > DSI_THR]  # F
tuned_other = cells[cells.sftf_tuned & ~cells.depth_tuned]  # D, E, grey
dir_tuned_other = tuned_other[tuned_other.fit_dsi > DSI_THR]  # F, grey
both = cells[cells.sftf_tuned & cells.rsof_tuned].reset_index(drop=True)  # B
depth_sftf = cells[cells.sftf_tuned & cells.depth_tuned].reset_index(drop=True)  # C

venn = dict(
    n_all=len(cells),
    n_depth=int(cells.depth_tuned.sum()),
    n_sftf=int(cells.sftf_tuned.sum()),
    n_both=int((cells.depth_tuned & cells.sftf_tuned).sum()),
)
venn["n_both_expected"] = (
    venn["n_depth"] * venn["n_sftf"] / venn["n_all"]
)  # if independent
stat_speed_of = bootstrap_spearman(both.pref_speed, both.preferred_of, both.session)
stat_speed_depth = bootstrap_spearman(
    depth_sftf.pref_speed, depth_sftf.preferred_depth * 100, depth_sftf.session
)

print(venn)
for name, st in [
    ("grating speed vs preferred OF", stat_speed_of),
    ("grating speed vs preferred depth", stat_speed_depth),
]:
    print(
        f"{name}: n = {st['n']}, rho = {st['rho']:.3f}, 95% CI {st['ci'].round(3)}, p = {st['p']:.3g}"
    )
print(
    f"D, E: {len(tuned)} depth-selective ({len(tuned_other)} other) cells; F: {len(dir_tuned)} ({len(dir_tuned_other)} other)"
)

In [ ]:
fig = plt.figure(figsize=(18 * CM, 12 * CM))
axes = {
    "A": fig.add_subplot(2, 3, 1),
    "B": fig.add_subplot(2, 3, 2),
    "C": fig.add_subplot(2, 3, 3),
    "D": fig.add_subplot(2, 3, 4),
    "E": fig.add_subplot(2, 3, 5, projection="polar"),
    "F": fig.add_subplot(2, 3, 6, projection="polar"),
}
plot_venn(
    axes["A"],
    venn["n_all"],
    venn["n_depth"],
    venn["n_sftf"],
    venn["n_both"],
    names=["Depth\nselective", "SF/TF\ntuned"],
    colors=["#2a78d6", "#1baf7a"],
)
log_scatter(
    axes["B"],
    both.pref_speed,
    both.preferred_of,
    stat_speed_of,
    "Preferred grating speed (deg/s)",
    "Preferred optic flow speed (deg/s)",
)
lims = [
    min(axes["B"].get_xlim()[0], axes["B"].get_ylim()[0]),
    max(axes["B"].get_xlim()[1], axes["B"].get_ylim()[1]),
]
axes["B"].plot(lims, lims, color="#8a8984", lw=0.8, ls="--", zorder=0)
log_scatter(
    axes["C"],
    depth_sftf.pref_speed,
    depth_sftf.preferred_depth * 100,
    stat_speed_depth,
    "Preferred grating speed (deg/s)",
    "Preferred virtual depth (cm)",
)

dsi_bins = np.linspace(0, 1, 21)
axes["D"].hist(
    tuned_other.fit_dsi,
    bins=dsi_bins,
    weights=np.full(len(tuned_other), 1 / len(tuned_other)),
    color="lightgrey",
    label=f"not depth selective (n = {len(tuned_other)})",
)
axes["D"].hist(
    tuned.fit_dsi,
    bins=dsi_bins,
    weights=np.full(len(tuned), 1 / len(tuned)),
    histtype="step",
    color="k",
    lw=2,
    label=f"depth selective (n = {len(tuned)})",
)
axes["D"].set_xlabel("Direction selectivity index")
axes["D"].set_ylabel("Proportion of neurons")
axes["D"].legend(fontsize=5, frameon=False, loc="upper center")
axes["D"].spines[["top", "right"]].set_visible(False)

for ax, col, period, n_bins, dep, oth, title in [
    (axes["E"], "fit_ori", 180, 4, tuned, tuned_other, "Preferred orientation (deg)"),
    (
        axes["F"],
        "fit_dir",
        360,
        8,
        dir_tuned,
        dir_tuned_other,
        f"Preferred direction (deg), dsi > {DSI_THR}",
    ),
]:
    polar_hist(ax, oth[col].values, period, n_bins, color="lightgrey")
    polar_hist(ax, dep[col].values, period, n_bins, step=True, color="k", lw=2)
    ax.set_title(
        f"{title}\nblack line: depth selective (n = {len(dep)}), grey: other (n = {len(oth)})",
        fontsize=6,
    )

for letter, ax in axes.items():
    ax.text(
        -0.15,
        1.08,
        letter,
        transform=ax.transAxes,
        fontweight="bold",
        fontsize=9,
        va="bottom",
    )
fig.tight_layout()
style.savefig(SAVE_ROOT / "fig_sftf_summary.pdf", fig=fig, bbox_inches="tight")

## Example neurons

Layout of the thesis example figure (`revisions/sftf/sftf_plots.py`): depth tuning and running speed / optic flow
tuning matrix (closed-loop depth protocol), then mean grating responses (left) and the fitted model (right) for the 8
directions around a polar plot. One depth-selective and one non-depth-selective SF/TF-tuned cell, both
`iscell`. ROI 544 is the thesis example; the thesis depth-selective example (ROI 229 of the same session)
touches the edge of the registered field of view and is not `iscell`.

In [ ]:
from v1_depth_map.figure_utils import sftf
from v1_depth_map.revisions.sftf import sftf_plots

# (session, roi): y-axis rounding of the depth tuning curve
EXAMPLES = {
    ("PZAH10.2f_S20230914", 447): dict(
        ylim_precision_base=0.1, ylim_precision=2
    ),  # depth selective
    ("PZAH10.2f_S20230817", 544): dict(
        ylim_precision_base=0.1, ylim_precision=2
    ),  # not depth selective
}
print(
    cells.set_index(["session", "roi"])
    .loc[
        list(EXAMPLES),
        [
            "depth_tuned",
            "preferred_depth",
            "sftf_tuned",
            "fit_sftf_test_rsq",
            "fit_dsi",
            "fit_dir",
        ],
    ]
    .round(2)
)

example_data = {
    s: sftf_plots.load_example_session(flm, s) for s in {s for s, _ in EXAMPLES}
}
fig = plt.figure(figsize=(18 * CM, 18 * CM))
for i, ((session, roi), ylim) in enumerate(EXAMPLES.items()):
    sftf_plots.plot_example_neuron(
        fig,
        example_data[session],
        roi,
        top=1 - i * 0.65,
        fontsize_dict={"title": 7, "label": 7, "tick": 5, "legend": 5},
        **ylim,
    )
style.savefig(SAVE_ROOT / "fig_sftf.pdf", fig=fig, bbox_inches="tight")

Additional depth-selective examples: one near-, one mid- and one far-preferring cell. Chosen by eye among the
top candidates (combined rank of held-out depth R², SF/TF R² and RS/OF R²) of `iscell` cells that are depth,
SF/TF and RS/OF tuned, with preferred depth 5-25 cm, 40-150 cm and 2-6 m.

In [ ]:
# (session, roi): y-axis rounding of the depth tuning curve
DEPTH_EXAMPLES = {
    "near": ("PZAH10.2d_S20230922", 102),
    "mid": ("PZAH10.2d_S20230922", 558),
    "far": ("PZAH10.2f_S20230914", 115),
}
# RS/OF matrix running-speed axis up to 100 cm/s (default 316): < 1% of closed-loop frames are faster
RSOF_LOG_RANGE = dict(
    rs_bin_log_min=0,
    rs_bin_log_max=2,
    rs_bin_num=5,
    of_bin_log_min=-1.5,
    of_bin_log_max=3.5,
    of_bin_num=11,
    log_base=10,
)
print(
    cells.set_index(["session", "roi"])
    .loc[
        list(DEPTH_EXAMPLES.values()),
        [
            "preferred_depth",
            "fit_sftf_test_rsq",
            "rsof_test_rsq",
            "preferred_of",
            "pref_speed",
            "fit_dsi",
        ],
    ]
    .round(2)
)

for s, _ in DEPTH_EXAMPLES.values():
    if s not in example_data:
        example_data[s] = sftf_plots.load_example_session(flm, s)
for band, (session, roi) in DEPTH_EXAMPLES.items():
    fig = plt.figure(
        figsize=(18 * CM, 18 * CM)
    )  # one band, same scale as the examples above
    sftf_plots.plot_example_neuron(
        fig,
        example_data[session],
        roi,
        top=1,
        fontsize_dict={"title": 7, "label": 7, "tick": 5, "legend": 5},
        ylim_precision_base=0.1,
        ylim_precision=2,
        rsof_log_range=RSOF_LOG_RANGE,
        rsof_vmax="max",  # default leaves out the last RS column and OF row
    )
    fig.text(0.02, 1.15, f"{band}: {session} ROI {roi}", fontsize=7, fontweight="bold")
    style.savefig(
        SAVE_ROOT / f"fig_sftf_examples_{band}.pdf", fig=fig, bbox_inches="tight"
    )

More mid-preferring examples (preferred depth 0.3-2 m), chosen by eye among the top candidates (combined rank of
held-out depth R² and SF/TF R²) of `iscell`, depth and SF/TF tuned cells. ROI 428 has the cleanest SF/TF tuning but
shallow depth modulation.

In [ ]:
MID_EXAMPLES = [
    ("PZAH10.2f_S20230817", 62),
    ("PZAH10.2d_S20230922", 562),
    ("PZAH10.2d_S20230920", 428),
]
print(
    cells.set_index(["session", "roi"])
    .loc[
        MID_EXAMPLES,
        [
            "preferred_depth",
            "fit_sftf_test_rsq",
            "rsof_test_rsq",
            "preferred_of",
            "pref_speed",
            "fit_dsi",
        ],
    ]
    .round(2)
)

for s, _ in MID_EXAMPLES:
    if s not in example_data:
        example_data[s] = sftf_plots.load_example_session(flm, s)
for session, roi in MID_EXAMPLES:
    fig = plt.figure(figsize=(18 * CM, 18 * CM))
    sftf_plots.plot_example_neuron(
        fig,
        example_data[session],
        roi,
        top=1,
        fontsize_dict={"title": 7, "label": 7, "tick": 5, "legend": 5},
        ylim_precision_base=0.1,
        ylim_precision=2,
        rsof_log_range=RSOF_LOG_RANGE,
        rsof_vmax="max",
    )
    fig.text(0.02, 1.15, f"mid: {session} ROI {roi}", fontsize=7, fontweight="bold")
    style.savefig(
        SAVE_ROOT / f"fig_sftf_examples_mid_{session}_{roi}.pdf",
        fig=fig,
        bbox_inches="tight",
    )

## Population SFTF tuning by depth preference (Near, Mid, Far)

Average SFTF response across cells split into 3 depth-preference bins with fixed bounds:
Near (< 20 cm), Mid (20–100 cm), Far (> 100 cm).
Each population is plotted in the same 8-direction + polar layout as the single-neuron examples,
showing both mean $\Delta F/F$ and normalized responses, as well as collapsed 2D marginals and 1D curves.


In [ ]:
# Extract and average SFTF data for the 3 depth populations (Fixed bounds: <20 cm, 20–100 cm, >100 cm)
pop_data = sftf_plots.extract_sftf_population_data(depth_sftf, CACHE)

# Raw (mean ΔF/F)
fig_f_raw = sftf_plots.plot_sftf_population_tuning(
    pop_data, strategy="fixed", metric="raw"
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_raw.pdf", fig=fig_f_raw, bbox_inches="tight"
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_raw.png",
    fig=fig_f_raw,
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
# Normalized (peak = 1 per cell)
fig_f_norm = sftf_plots.plot_sftf_population_tuning(
    pop_data, strategy="fixed", metric="norm"
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_norm.pdf",
    fig=fig_f_norm,
    bbox_inches="tight",
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_norm.png",
    fig=fig_f_norm,
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
# --- Direction-collapsed 2D marginals and 1D SF/TF tuning curves ---
# Raw (mean ΔF/F)
fig_m_raw = sftf_plots.plot_sftf_population_marginals(
    pop_data, strategy="fixed", metric="raw"
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_marginals_raw.pdf",
    fig=fig_m_raw,
    bbox_inches="tight",
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_marginals_raw.png",
    fig=fig_m_raw,
    bbox_inches="tight",
    dpi=300,
)

# Normalized (peak = 1 per cell)
fig_m_norm = sftf_plots.plot_sftf_population_marginals(
    pop_data, strategy="fixed", metric="norm"
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_marginals_norm.pdf",
    fig=fig_m_norm,
    bbox_inches="tight",
)
style.savefig(
    SAVE_ROOT / "fig_sftf_populations_fixed_marginals_norm.png",
    fig=fig_m_norm,
    bbox_inches="tight",
    dpi=300,
)

## Exploratory: SF / TF responses by running speed

Mean dF/F per SF, TF and direction (data only, no fit) computed from running imaging frames (RS > 1 cm/s,
dF/F of the same frame, `FRAME_LAG = 0`, as in `precompute_data/sftf_running_models.py`), split into the treadmill
running-speed bins of figure 3 (one octave around each motor speed) with the two slowest merged and the first bin starting at 5 cm/s (the still threshold): 5.0-11.3, 11.3-22.6,
22.6-45.3 and 45.3-90.5 cm/s (frames outside are dropped). Colour scale shared across the bins of a cell.
Blank squares: no frame of that condition in the bin (whole trials can fall in one speed bin). Cells: the two
example neurons, and the three with the largest held-out gain of the TF-dependent model in
`sftf_running_models` (S20230817 ROIs 244 and 214, S20230914 ROI 440).

In [ ]:
from v1_depth_map.precompute_data import sftf_running_models

SPEED_EXAMPLES = [
    ("PZAH10.2f_S20230924", 430),  # sharply SF tuned (0.08 cpd), TF rising to 16 Hz
    ("PZAH10.2d_S20230920", 429),  # SF (0.02 cpd) and TF (4 Hz) tuned
    ("PZAH10.2f_S20230817", 249),  # SF (0.04 cpd) and TF (4 Hz) tuned
    ("PZAH10.2f_S20230817", 244),
    ("PZAH10.2f_S20230817", 214),
]
# running-speed bins used by every plot split by speed: the treadmill bins of figure 3
# (figure_utils.treadmill.compute_treadmill_rsof_bins on PZAG17.3a_S20250402, one octave around each motor speed,
# 4, 8, 15, 30, 61 cm/s; rs_bins[2:] as in panels G-I), with the two slowest merged, first bin from 5 cm/s; frames outside are dropped
SPEED_EDGES_CM_S = [5, 11.31, 22.63, 45.25, 90.51]  # first bin from 5 cm/s, the still threshold (STILL_THR)
# speed-binned TF (SF) curves: SF (TF) / direction pairs >= this fraction of the best pair
TF_SELECT_THRESHOLD = 0.5


def example_frames(session, rois):
    # running frames of a session and the dF/F of `rois` only, cached
    path = CACHE / f"{session}_speed_frames_lag{FRAME_LAG}.pickle"
    if path.exists() and not REBUILD:
        cached = pd.read_pickle(path)
        if set(rois) <= set(cached["dff"]):
            return cached
    X, dff, _, log_rs_median = sftf_running_models.session_frames(
        flm, session, trace="dff", lag=FRAME_LAG
    )
    cached = dict(X=X, log_rs_median=log_rs_median, dff={r: dff[:, r] for r in rois})
    pd.to_pickle(cached, path)
    return cached


speed_frames = {
    s: example_frames(s, [r for s2, r in SPEED_EXAMPLES if s2 == s])
    for s in {s for s, _ in SPEED_EXAMPLES}
}
speed_binned = {}
for session, roi in SPEED_EXAMPLES:
    f = speed_frames[session]
    binned = sftf.speed_binned_responses(
        f["X"], f["dff"][roi], f["log_rs_median"], edges_cm_s=SPEED_EDGES_CM_S
    )
    speed_binned[(session, roi)] = binned
    fig = plt.figure(figsize=(18 * CM, 5.5 * CM), dpi=600)
    sftf_plots.plot_sftf_by_speed(
        fig,
        binned,
        title=f"{session} ROI {roi}",
        fontsize_dict={"title": 7, "label": 6, "tick": 5, "legend": 5},
    )
    style.savefig(
        SAVE_ROOT / f"sftf_by_speed_{session}_{roi}.pdf", fig=fig, bbox_inches="tight"
    )

Running-speed distribution during the gratings, per session: running frames (RS > 1 cm/s, the frames used above),
log-spaced bins, proportion of frames. Dotted lines: speed bin edges.

In [ ]:
rs_bins = np.logspace(np.log10(1), np.log10(150), 50)
fig, ax = plt.subplots(figsize=(9 * CM, 5 * CM), dpi=600)
colors = plt.get_cmap("tab10")(np.arange(len(SESSIONS)))
for session, color in zip(SESSIONS, colors):
    d = np.load(SAVE_ROOT / "running_models" / "frames" / f"{session}.npz")
    rs_cm_s = np.exp(d["X"][3] + float(d["log_rs_median"])) * 100
    counts, _ = np.histogram(rs_cm_s, bins=rs_bins)
    centres = np.sqrt(rs_bins[1:] * rs_bins[:-1])
    ax.plot(
        centres,
        counts / counts.sum(),
        color=color,
        lw=1,
        label=f"{session} (median {np.median(rs_cm_s):.0f} cm/s, max {np.max(rs_cm_s):.0f} cm/s)",
    )
ax.set_xlabel("Running speed (cm/s)", fontsize=6)
ax.set_ylabel("Proportion of running frames", fontsize=6)
ax.tick_params(labelsize=5)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=5, frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
style.savefig(
    SAVE_ROOT / "running_speed_distribution.pdf", fig=fig, bbox_inches="tight"
)

## Running-speed dependence of SF / TF preferences: model comparison

Frame-level fits of `precompute_data/sftf_running_models.py` (NEMO, all 3012 `iscell` cells of the six
sessions, whether or not they are SF/TF tuned in the trial-averaged fit; `prepare --all-cells`). The running-speed
dependence of the amplitude is a Gaussian in log running speed, parameterised and bounded as the RS/OF fits of
figure 2 (`fit_gaussian_blob.fit_rs_of_tuning`):

$$\ln A(r) = \ell - \frac{(r - r_0)^2}{2 \sigma^2}, \qquad \sigma^2 = e^{\log \sigma^2} + 0.25$$

with $r = \ln \mathrm{RS} - \mathrm{median}(\ln \mathrm{RS})$, preferred running speed $\mathrm{RS}_{med} \cdot e^{r_0}$
between 0.5 cm/s and 5 m/s, and $\log \sigma^2$ free. Models:
- **base**: SF, TF and direction tuning, no running-speed dependence;
- **rsg**: base with the running-speed Gaussian on the amplitude;
- **rsg_tf**: rsg with a TF preference shift, $\log \mathrm{TF}_0 = t_0 + a_{tf} \cdot r$;
- **rsg_sf**: rsg with an SF preference shift, $\log \mathrm{SF}_0 = s_0 + a_{sf} \cdot r$.

Running frames only (RS > 1 cm/s), dF/F of the same frame (`FRAME_LAG = 0`, as the RS/OF fits; lag sweep in
`revisions/rf_spike_tau/lag_choice.ipynb`); held-out R² leaving one recording out.
Cells restricted, as in figure 2E, to those whose held-out R² is above the empirical-null threshold (mirrored
negative tail, 95th percentile, one threshold per model) in at least one of the four models. Shift slopes are
reported for the cells where the model with that shift is significant. Depth-selective vs other cells: depth
labels permuted within session (one-sided, depth > other).

In [ ]:
# running-speed model comparison statistics
RS_MODELS = ["base", "rsg", "rsg_tf", "rsg_sf"]
RS_MODEL_LABELS = ["SF, TF", "+ RS", "+ RS + TF shift", "+ RS + SF shift"]
pickle_files = [
    f
    for f in sorted((SAVE_ROOT / "running_models").glob("*_sftf_running_models.pickle"))
    if not f.name.startswith(".")
]
rs_fits = pd.concat([pd.read_pickle(f) for f in pickle_files], ignore_index=True)
rs_fits = rs_fits.merge(
    cells[["session", "roi", "depth_tuned"]],
    on=["session", "roi"],
    how="left",
    validate="one_to_one",
)
missing = {m: int(rs_fits[f"{m}_test_rsq"].isna().sum()) for m in RS_MODELS}
rs_thresholds = common_utils.add_rsq_significance(
    rs_fits, {m: f"{m}_test_rsq" for m in RS_MODELS}, verbose=False
)
rs_fits["rs_sig"] = rs_fits[[f"{m}_test_rsq_sig" for m in RS_MODELS]].any(axis=1)
rs_fits["best_model"] = (
    rs_fits[[f"{m}_test_rsq" for m in RS_MODELS]]
    .idxmax(axis=1)
    .str.replace("_test_rsq", "")
)
rs_fits["dR2_tf_over_rsg"] = rs_fits.rsg_tf_test_rsq - rs_fits.rsg_test_rsq
rs_fits["dR2_sf_over_rsg"] = rs_fits.rsg_sf_test_rsq - rs_fits.rsg_test_rsq
rs_fits["best_tf_shift"] = rs_fits.best_model == "rsg_tf"
rs_fits["best_sf_shift"] = rs_fits.best_model == "rsg_sf"


def bootstrap_median(values, sessions, n_boots=N_BOOTS_EXPLORATORY):
    # median, 95% CI and p (vs 0) from a hierarchical bootstrap over mice then sessions
    df = (
        pd.DataFrame(dict(x=np.asarray(values), session=np.asarray(sessions)))
        .dropna()
        .reset_index(drop=True)
    )
    with contextlib.redirect_stderr(io.StringIO()):
        _, dist = common_utils.hierarchical_bootstrap_stats(
            df, n_boots, ["x"], ["mouse", "session"], plot=False
        )
    return dict(
        n=len(df),
        median=float(np.median(df.x)),
        ci=np.percentile(dist[:, 0], [2.5, 97.5]),
        p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
        frac_pos=float(np.mean(df.x > 0)),
    )


N_PERM = 10000
perm_rng = np.random.default_rng(0)


def depth_permutation(df, col, stat, strata, n=N_PERM):
    # difference (depth - other) of `stat` and one-sided p, depth labels permuted within `strata`
    df = df[df[col].notna()]
    vals, labels = df[col].to_numpy(float), df.depth_tuned.to_numpy(bool)
    obs = stat(vals[labels]) - stat(vals[~labels])
    keys = df[strata].astype(str).agg("_".join, axis=1).to_numpy()
    groups = [np.flatnonzero(keys == g) for g in np.unique(keys)]
    null = np.empty(n)
    for i in range(n):
        lab = labels.copy()
        for idx in groups:
            lab[idx] = perm_rng.permutation(lab[idx])
        null[i] = stat(vals[lab]) - stat(vals[~lab])
    return obs, (1 + np.sum(null >= obs)) / (n + 1)


rs_set = rs_fits[rs_fits.rs_sig]
RS_GROUPS = {
    "all": rs_set,
    "depth": rs_set[rs_set.depth_tuned],
    "other": rs_set[~rs_set.depth_tuned],
}
print(
    f"{len(rs_fits)} cells ({missing} without fit); {len(rs_set)} significant in at least one model "
    f"(thresholds {({m: round(float(v), 4) for m, v in rs_thresholds.items()})})"
)
best_props = pd.DataFrame(
    {
        k: v.best_model.value_counts(normalize=True).reindex(RS_MODELS)
        for k, v in RS_GROUPS.items()
    }
)
display(best_props.round(3))

# shift slopes in the cells where the model with that shift is significant
SHIFT_COLS = {"rsg_tf_a_tf": "rsg_tf", "rsg_sf_a_sf": "rsg_sf"}
shift_sets = {}
for col, model in SHIFT_COLS.items():
    sig = rs_fits[rs_fits[f"{model}_test_rsq_sig"]]
    shift_sets[("all", col)] = sig
    shift_sets[("depth", col)] = sig[sig.depth_tuned]
    shift_sets[("other", col)] = sig[~sig.depth_tuned]
rs_stats, rows = {}, []
for col, stat, label, source in [
    ("rsg_tf_a_tf", np.median, "TF shift a_tf (rsg_tf significant)", "shift"),
    ("rsg_sf_a_sf", np.median, "SF shift a_sf (rsg_sf significant)", "shift"),
    ("dR2_tf_over_rsg", np.median, "held-out R² rsg_tf - rsg", "rs"),
    ("dR2_sf_over_rsg", np.median, "held-out R² rsg_sf - rsg", "rs"),
    ("best_tf_shift", np.mean, "best model rsg_tf", "rs"),
    ("best_sf_shift", np.mean, "best model rsg_sf", "rs"),
]:
    sets = (
        {g: shift_sets[(g, col)] for g in RS_GROUPS} if source == "shift" else RS_GROUPS
    )
    row = dict(measure=label)
    for g, sub in sets.items():
        row[f"{g}_n"] = int(sub[col].notna().sum())
        row[g] = stat(sub[col].dropna())
        if stat is np.median:
            rs_stats[(g, col)] = bootstrap_median(sub[col], sub.session)
            row[f"{g}_p_vs_0"] = rs_stats[(g, col)]["p"]
    row["p_depth_gt_other"] = depth_permutation(sets["all"], col, stat, ["session"])[1]
    rows.append(row)
rs_stats_table = pd.DataFrame(rows).set_index("measure")
display(rs_stats_table.round(4))

In [ ]:
# Model comparison plots: SF, TF (= base) vs + RS (rsg), + RS + TF shift (rsg_tf), + RS + SF shift (rsg_sf)
import seaborn as sns
from cottage_analysis.plotting import rsof_plots

FONTSIZE_DICT_SMALL = {"title": 6, "label": 6, "tick": 5}


def fmt_boot_p(p, n_boots=N_BOOTS_EXPLORATORY):
    return f"p < {1 / n_boots:.0e}" if p == 0 else f"p = {p:.2g}"


SHIFT_COLORS = {"rsg_tf": "#3182bd", "rsg_sf": "#756bb1"}
SHIFT_ROWS = {  # rows of rs_stats_table
    "rsg_tf_a_tf": "TF shift a_tf (rsg_tf significant)",
    "rsg_sf_a_sf": "SF shift a_sf (rsg_sf significant)",
}
fig = plt.figure(figsize=(22 * CM, 7 * CM), dpi=600)
gs = fig.add_gridspec(2, 4, width_ratios=[0.7, 0.7, 1.1, 1.1])
axes = [fig.add_subplot(gs[:, k]) for k in [0, 1]]  # A
axes_b = [fig.add_subplot(gs[k, 2]) for k in [0, 1]]  # B: TF (top), SF (bottom)
axes += [fig.add_subplot(gs[:, 3])]  # C
fig.tight_layout()

# A: proportion of cells best explained by each model (as figure 2E), depth selective or not
for ax, (name, group) in zip(
    axes[:2], [("depth selective", "depth"), ("not depth selective", "other")]
):
    pos = ax.get_position()
    ax.remove()
    sel = RS_GROUPS[group].assign(iscell=1)
    sel = sel.rename(
        columns={f"{m}_test_rsq": f"rsof_test_rsq_closedloop_{m}" for m in RS_MODELS}
    )
    rsof_plots.plot_r2_comparison(
        fig=fig,
        neurons_df=sel,
        models=RS_MODELS,
        labels=RS_MODEL_LABELS,
        plot_type="bar",
        markersize=4,
        alpha=0.5,
        color="grey",
        plot_x=pos.x0,
        plot_y=pos.y0,
        plot_width=pos.width,
        plot_height=pos.height,
        reference_model="base",
        fontsize_dict=FONTSIZE_DICT_SMALL,
    )
    ax_a = plt.gca()
    ax_a.set_ylim([-0.02, 1.02])
    sns.despine(ax=ax_a, offset=5, trim=True)
    ax_a.tick_params(axis="x", rotation=90)
    ax_a.set_title(
        f"{name}\nn = {len(sel)} cells, {sel.session.nunique()} sessions", fontsize=6
    )
    if group == "other":
        ax_a.set_ylabel("")

# B: TF (top) and SF (bottom) shift slopes, cells where that model is significant;
# other cells light grey filled, depth-selective cells black outline
bins = np.linspace(-1, 1.5, 21)  # values outside are clipped into the edge bins
for ax, (col, label) in zip(
    axes_b,
    [("rsg_tf_a_tf", "TF shift a_tf (+ RS)"), ("rsg_sf_a_sf", "SF shift a_sf (+ RS)")],
):
    for group, histtype, color, lw in [
        ("other", "stepfilled", "lightgrey", 0),
        ("depth", "step", "k", 2),
    ]:
        v = shift_sets[(group, col)][col].dropna()
        st = rs_stats[(group, col)]
        ax.hist(
            np.clip(v, bins[0], bins[-1]),
            bins=bins,
            weights=np.ones(len(v)) / len(v),
            histtype=histtype,
            color=color,
            lw=lw,
            label=f"{group} (n = {len(v)}): med {st['median']:.2f}\n{fmt_boot_p(st['p'])}",
        )
        ax.axvline(
            st["median"], color="grey" if group == "other" else "k", lw=0.8, ls="--"
        )
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(
        f"{label}; depth > other p = {rs_stats_table.loc[SHIFT_ROWS[col], 'p_depth_gt_other']:.2g}",
        fontsize=6,
    )
    ax.set_ylabel("Proportion of cells")
    ax.legend(frameon=False, fontsize=4.5, loc="upper right")
axes_b[1].set_xlabel("Shift slope (log freq. per log RS)")

# C: what the preference shift adds to the running-speed model, depth selective (solid) or not (dashed)
ax = axes[2]
for col, model in [("dR2_tf_over_rsg", "rsg_tf"), ("dR2_sf_over_rsg", "rsg_sf")]:
    for group, ls in [("depth", "-"), ("other", "--")]:
        g = np.sort(RS_GROUPS[group][col].dropna())
        st = rs_stats[(group, col)]
        ax.step(
            g,
            np.arange(1, len(g) + 1) / len(g),
            where="post",
            color=SHIFT_COLORS[model],
            ls=ls,
            lw=1.2,
            label=f"{model.split('_')[1].upper()} shift, {group}: "
            f"med {st['median']:.1e}, {fmt_boot_p(st['p'])}",
        )
ax.axvline(0, color="k", lw=0.5)
ax.set_xlim([-0.003, 0.005])
ax.set_xlabel("Held-out R² improvement over + RS")
ax.set_ylabel("Cumulative proportion")
ax.legend(frameon=False, fontsize=4.5)
for ax in axes[2:] + axes_b:
    ax.spines[["top", "right"]].set_visible(False)

style.savefig(SAVE_ROOT / "running_speed_models.pdf", fig=fig, bbox_inches="tight")
print(f"Saved to {SAVE_ROOT / 'running_speed_models.pdf'}")

The gain vs preferred running speed comparison (`gain` / `rsg` models) is in
`revisions/sftf/running_speed_gain_vs_preferred_rs.ipynb` (personal notes, not for the paper).

TF and SF tuning per running-speed bin for example cells (chosen by eye among strong, well fit cells with a
significant model-free TF shift in an earlier version of this notebook), depth selective or not; titles give the
TF shift slope of the RS + TF shift model. For each cell, the SF (TF) / direction pairs whose mean response (over
TF (SF) and all running frames in 5.0-90.5 cm/s) is at least `TF_SELECT_THRESHOLD` of the best pair are kept (same
pairs for all bins); the curve is the mean over these pairs of the response at each TF (SF), ± SEM across pairs.
Top: TF curves; bottom: SF curves; the 4 speed bins of the plots above.

In [ ]:
# example cells (chosen by eye among strong, well fit cells with a TF shift)
TF_SHIFT_EXAMPLES = [
    ("PZAH10.2f_S20230924", 95),
    ("PZAH10.2f_S20230914", 54),
    ("PZAH10.2f_S20230817", 259),
    ("PZAH10.2f_S20230817", 214),
    ("PZAH10.2f_S20230817", 244),
]
fig, axes = plt.subplots(2, len(TF_SHIFT_EXAMPLES), figsize=(18 * CM, 9 * CM))
for j, (session, roi) in enumerate(TF_SHIFT_EXAMPLES):
    cell = rs_fits.set_index(["session", "roi"]).loc[(session, roi)]
    d = np.load(FRAMES_DIR / f"{session}.npz")
    col = int(np.flatnonzero(d["rois"] == roi)[0])
    binned = sftf.speed_binned_responses(
        d["X"], d["dff"][:, col], float(d["log_rs_median"]), edges_cm_s=SPEED_EDGES_CM_S
    )
    for i, (variable, pairs) in enumerate([("tf", "SF"), ("sf", "TF")]):
        curves, kept = sftf.tf_curves_by_speed(
            binned, threshold=TF_SELECT_THRESHOLD, variable=variable
        )
        sftf_plots.plot_tf_curves(axes[i, j], curves)
        if variable == "sf":
            axes[i, j].tick_params(axis="x", rotation=45)
        axes[i, j].set_title(
            (
                f"{session.split('_')[1]} ROI {roi}\n"
                f"{'depth selective' if cell.depth_tuned else 'not depth selective'}\n"
                f"a_tf {cell.rsg_tf_a_tf:+.2f} (rsg_tf held-out R² {cell.rsg_tf_test_rsq:.3f})\n"
                if i == 0
                else ""
            )
            + f"{len(kept)} {pairs}/direction pairs",
            fontsize=5.5,
        )
for i in range(2):
    axes[i, -1].legend(
        title="Running speed",
        fontsize=5,
        title_fontsize=5,
        frameon=False,
        bbox_to_anchor=(1, 1),
        loc="upper left",
    )
fig.tight_layout()
style.savefig(SAVE_ROOT / "tf_curves_by_speed.pdf", fig=fig, bbox_inches="tight")

### Population TF and SF tuning per running-speed bin

TF curves per speed bin (as the example curves above: SF / direction pairs ≥ `TF_SELECT_THRESHOLD` of the
best pair, same pairs for all bins), for all cells significant on the base SF/TF frame-level fit
(`base_test_rsq` above the empirical threshold). Each cell's preferred TF is the peak of its data curve averaged
over the speed bins; TF is expressed in octaves from that peak, and each cell's curves are divided by the value
of that peak (so gain differences between bins are kept). Mean ± SEM across cells; points with < 10 cells not
shown. Bottom row: the same for SF (TF / direction pairs ≥ threshold). The peak is taken from the same data, so
the value at 0 is biased upwards by noise; this bias is the same for all speed bins.

In [ ]:
MIN_CELLS_PER_POINT = 10
POP_VARIABLES = {"tf": "TF", "sf": "SF"}
# cells significant on the base SF/TF frame-level fit
pop_cells = rs_fits[rs_fits.base_test_rsq_sig]
pop_means, pop_n = {}, pop_cells.depth_tuned.value_counts()
for var, var_label in POP_VARIABLES.items():
    pop = []
    for session, sc in pop_cells.groupby("session"):
        d = np.load(FRAMES_DIR / f"{session}.npz")
        cols = [int(np.flatnonzero(d["rois"] == r)[0]) for r in sc.roi]
        curves, values, speed_labels = sftf.population_tf_curves(
            d["X"],
            d["dff"][:, cols],
            float(d["log_rs_median"]),
            SPEED_EDGES_CM_S,
            threshold=TF_SELECT_THRESHOLD,
            variable=var,
        )  # (cell, speed bin, frequency)
        # preferred frequency: peak of the curve averaged over speed bins; normalised by that peak
        mean_curve = np.nanmean(curves, axis=1)
        peak = np.nanargmax(mean_curve, axis=1)
        peak_value = mean_curve[np.arange(len(cols)), peak]
        for k, (_, cell) in enumerate(sc.iterrows()):
            for b, label in enumerate(speed_labels):
                pop.append(
                    pd.DataFrame(
                        dict(
                            depth_tuned=cell.depth_tuned,
                            speed_bin=b,
                            speed=label,
                            delta=np.arange(len(values)) - peak[k],
                            resp=curves[k, b] / peak_value[k],
                        )
                    )
                )
    pop = pd.concat(pop, ignore_index=True)
    pop_means[var] = (
        pop.groupby(["depth_tuned", "speed_bin", "speed", "delta"])
        .resp.agg(["mean", "sem", "count"])
        .reset_index()
    )
print(
    f"{len(pop_cells)} cells significant on the base fit, {pop_cells.depth_tuned.sum()} depth selective"
)

fig, axes = plt.subplots(2, 2, figsize=(14 * CM, 10 * CM), sharey=True)
colors = plt.get_cmap("viridis")(np.linspace(0, 0.9, len(speed_labels)))
for i, (var, var_label) in enumerate(POP_VARIABLES.items()):
    for j, (name, is_depth) in enumerate(
        [("depth selective", True), ("not depth selective", False)]
    ):
        ax = axes[i, j]
        m = pop_means[var]
        sub = m[(m.depth_tuned == is_depth) & (m["count"] >= MIN_CELLS_PER_POINT)]
        for b, (label, color) in enumerate(zip(speed_labels, colors)):
            c = sub[sub.speed_bin == b].sort_values("delta")
            ax.errorbar(
                c.delta,
                c["mean"],
                yerr=c["sem"],
                color=color,
                lw=1,
                marker="o",
                ms=2,
                capsize=0,
                label=label,
            )
        ax.set_title(f"{var_label}, {name} (n = {pop_n.get(is_depth, 0)})", fontsize=6)
        ax.axvline(0, color="k", lw=0.5, ls=":")
        ax.set_xlabel(f"{var_label} relative to peak {var_label} (octaves)", fontsize=6)
        ax.tick_params(labelsize=5)
        ax.spines[["top", "right"]].set_visible(False)
    axes[i, 0].set_ylabel("ΔF/F / peak", fontsize=6)
axes[0, 1].legend(
    title="Running speed",
    fontsize=5,
    title_fontsize=5,
    frameon=False,
    bbox_to_anchor=(1, 1),
    loc="upper left",
)
fig.tight_layout()
style.savefig(
    SAVE_ROOT / "population_tf_curves_by_speed.pdf", fig=fig, bbox_inches="tight"
)

## Exploratory: SF / TF preferences vs depth and speed preferences

Preferred SF, TF and grating speed (TF / SF) against preferred virtual depth (depth-selective, SF/TF-tuned
cells) and against the preferred optic flow and running speed of the RS/OF fit (SF/TF- and RS/OF-tuned
cells, as in B and C). Points coloured by local density. Spearman ρ across cells with a hierarchical bootstrap
(mice, sessions, `N_BOOTS_EXPLORATORY` resamples). Not exported to the manuscript stats.

In [ ]:
# exploratory correlations
EXPLORE_X = {  # column: (label, transform)
    "fit_sf0": ("Preferred SF (cpd)", np.exp),
    "fit_tf0": ("Preferred TF (Hz)", np.exp),
    "pref_speed": ("Preferred grating speed (deg/s)", lambda v: v),
}
EXPLORE_Y = {  # column: (label, transform, selection)
    "preferred_depth": (
        "Preferred virtual depth (cm)",
        lambda v: v * 100,
        cells.depth_tuned & cells.sftf_tuned,
    ),
    "preferred_of": (
        "Preferred optic flow (deg/s)",
        lambda v: v,
        cells.sftf_tuned & cells.rsof_tuned,
    ),
    "preferred_rs": (
        "Preferred running speed (cm/s)",
        lambda v: v * 100,
        cells.sftf_tuned & cells.rsof_tuned,
    ),
}
explore = {}
for ycol, (_, yf, sel) in EXPLORE_Y.items():
    sub = cells[sel]
    for xcol, (_, xf) in EXPLORE_X.items():
        st = bootstrap_spearman(
            xf(sub[xcol]), yf(sub[ycol]), sub.session, n_boots=N_BOOTS_EXPLORATORY
        )
        explore[(xcol, ycol)] = st
        print(
            f"{xcol:10s} vs {ycol:15s}: n = {st['n']}, rho = {st['rho']:.2f}, bootstrap p = {st['p']:.2g}"
        )

In [ ]:
from cottage_analysis.plotting import plotting_utils

fig, axes = plt.subplots(len(EXPLORE_Y), len(EXPLORE_X), figsize=(17 * CM, 15 * CM))
for i, (ycol, (ylabel, yf, sel)) in enumerate(EXPLORE_Y.items()):
    sub = cells[sel]
    for j, (xcol, (xlabel, xf)) in enumerate(EXPLORE_X.items()):
        ax, st = axes[i, j], explore[(xcol, ycol)]
        plotting_utils.density_scatter(
            xf(sub[xcol]), yf(sub[ycol]), ax=ax, log_scale=True, bins=32, s=5
        )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.text(
            0.5,
            1.03,
            f"ρ = {st['rho']:.2f} (n = {st['n']}), ",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=6,
        )
        ax.text(
            0.5,
            1.03,
            f"p = {st['p']:.2g}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=6,
            color="green" if st["p"] < 0.05 else "red",
        )
        if i == len(EXPLORE_Y) - 1:
            ax.set_xlabel(xlabel)
        if j == 0:
            ax.set_ylabel(ylabel)
        ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(
    SAVE_ROOT / "exploratory_sftf_correlations.pdf", fig=fig, bbox_inches="tight"
)

## Proportions of responsive and selective cells

In [ ]:
# proportions per group and χ² depth tuned vs not
SELECTIVITY = ["responsive", "ori_selective", "dir_selective"]
counts = cells.groupby("group")[SELECTIVITY].agg(["sum", "mean"])
counts.insert(0, "n", cells.groupby("group").size())
display(counts)

per_session = cells.groupby(["session", "group"])[SELECTIVITY].mean().unstack("group")
display(per_session.round(2))

prop_tests = {}
for col in SELECTIVITY:
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(cells.group, cells[col]))
    prop_tests[col] = dict(chi2=chi2, p=p)
    print(f"{col:14s} depth tuned vs not: chi2 = {chi2:.1f}, p = {p:.2g}")

In [ ]:
labels = ["responsive", "orientation\nselective", "direction\nselective"]
fig, ax = plt.subplots(figsize=(8 * CM, 5 * CM))
width = 0.38
for i, (group, color) in enumerate(GROUPS.items()):
    x = np.arange(len(SELECTIVITY)) + (i - 0.5) * width
    ax.bar(
        x,
        counts.loc[group, [(c, "mean") for c in SELECTIVITY]].values,
        width=width * 0.92,
        color=color,
        label=group,
    )
    sess = per_session.xs(group, axis=1, level="group")[SELECTIVITY]
    for j, c in enumerate(SELECTIVITY):
        ax.scatter(
            np.full(len(sess), x[j]), sess[c], s=6, color="k", zorder=3, linewidths=0
        )
ax.set_xticks(np.arange(len(SELECTIVITY)), labels)
ax.set_ylabel("Proportion of cells")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
style.savefig(SAVE_ROOT / "proportions.pdf", fig=fig, bbox_inches="tight")

## Preferred orientation and direction

Orientation preference of orientation-selective cells and direction preference of direction-selective
cells, one bin per stimulus orientation / direction (the vector sum is continuous, but only 8 directions
were shown). Comparison: χ² on the binned counts.

In [ ]:
# binned preferences per group and χ² depth tuned vs not
def binned_counts(values, period, n_bins):
    # counts in bins centred on multiples of period / n_bins
    width = period / n_bins
    edges = np.arange(n_bins + 1) * width - width / 2
    return np.histogram((values + width / 2) % period - width / 2, bins=edges)[0], edges


PREFERENCES = {
    "orientation": dict(col="pref_ori", sel="ori_selective", period=180, n_bins=4),
    "direction": dict(col="pref_dir", sel="dir_selective", period=360, n_bins=8),
}
pref_counts, pref_tests = {}, {}
for name, d in PREFERENCES.items():
    pref_counts[name] = {
        group: binned_counts(
            cells.loc[(cells.group == group) & cells[d["sel"]], d["col"]].values,
            d["period"],
            d["n_bins"],
        )
        for group in GROUPS
    }
    table = np.array([c for c, _ in pref_counts[name].values()])
    chi2, p, _, _ = stats.chi2_contingency(table + 0.5)
    pref_tests[name] = dict(chi2=chi2, p=p, n=int(table.sum()))
    print(f"preferred {name}: {dict(zip(GROUPS, table.tolist()))}, chi2 p = {p:.2g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16 * CM, 5 * CM))
for ax, (name, d) in zip(axes, PREFERENCES.items()):
    for group, color in GROUPS.items():
        c, edges = pref_counts[name][group]
        ax.step(
            np.append(edges[:-1], edges[-1]),
            np.append(c, c[-1]) / c.sum(),
            where="post",
            color=color,
            lw=1.5,
            label=f"{group} (n={c.sum()})",
        )
    ax.set_title(
        f"Preferred {name} ({d['sel'].replace('_', ' ')})\nχ² p = {pref_tests[name]['p']:.2g}",
        fontsize=7,
    )
    ax.set_xticks(np.arange(0, d["period"] + 1, 45))
    ax.set_ylim(0, None)
    ax.set_xlabel(f"Preferred {name} (deg)")
    ax.set_ylabel("Proportion of cells")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=6)
style.savefig(
    SAVE_ROOT / "preferred_orientation_direction.pdf", fig=fig, bbox_inches="tight"
)

---
# Exploratory: TF × RS slope vs preferred grating direction and RS/OF tuning orientation

**Exploratory, not robust; nothing here is exported to the manuscript stats.** These cells ask whether the TF ×
RS slope $a_{tf}$ (RS + TF shift model) depends on the preferred grating direction or on the orientation of the
RS/OF 2D Gaussian (figure 3K). Checks done on 2026-10-14 (scratch scripts, not in this notebook):
- *RS/OF orientation near 45°*: the association with $a_{tf}$ depends on the selection (ρ = −0.21, p = 0.04 with
  the rsg_tf significance filter below; ρ = −0.06, p = 0.2 without it). The model-free TF shift shows a modest
  trend (ρ ≈ −0.18), mostly from one mouse, after several variants were tried.
- *Preferred motion along the local optic flow* (front to back, `Angle` + 180° ≈ 0° in map coordinates): larger
  $a_{tf}$ for with-flow than against-flow cells (0.18 vs 0.04, p = 0.005, 6/6 sessions, in depth-selective and
  other cells alike), but not seen with the model-free TF shift; a running-related horizontal eye drift could
  produce it (not checked).


### TF × RS slope against preferred grating direction

Preferred direction (angle, `rsg_tf_alpha0` = grating `Angle`, i.e. wave vector; motion is towards angle + 180°)
and direction selectivity (radius) of the RS + TF shift model, coloured by $a_{tf}$; cells with a significant
rsg_tf fit.


In [ ]:
# preferred direction (angle) and direction selectivity (radius) of the RS + TF shift model,
# coloured by its TF x RS slope a_tf; cells with a significant rsg_tf fit
from matplotlib.colors import TwoSlopeNorm

A_TF_LIM = 1.0  # colour scale +- this (clipped)
sub = rs_fits[rs_fits.rsg_tf_test_rsq_sig].dropna(
    subset=["rsg_tf_alpha0", "rsg_tf_dsi", "rsg_tf_a_tf"]
)
sub = sub.iloc[
    np.argsort(np.abs(sub.rsg_tf_a_tf.to_numpy()))
]  # strongest slopes on top

fig, axes = plt.subplots(
    1, 2, figsize=(14 * CM, 7 * CM), subplot_kw=dict(projection="polar"), dpi=600
)
for ax, (label, sel) in zip(
    axes,
    [
        ("depth tuned", sub.depth_tuned == True),
        ("not depth tuned", sub.depth_tuned != True),
    ],
):
    d = sub[sel]
    sc = ax.scatter(
        np.mod(d.rsg_tf_alpha0, 2 * np.pi),
        d.rsg_tf_dsi,
        c=d.rsg_tf_a_tf,
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vcenter=0, vmin=-A_TF_LIM, vmax=A_TF_LIM),
        s=10,
        linewidths=0,
    )
    ax.set_ylim(0, 1)
    ax.set_yticks([0.5, 1])
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45)))
    ax.tick_params(labelsize=5)
    ax.set_title(f"{label} (n = {len(d)})", fontsize=7)
cbar = fig.colorbar(sc, ax=axes, pad=0.08)
cbar.set_label("TF × RS slope $a_{tf}$", fontsize=6)
cbar.ax.tick_params(labelsize=5)
style.savefig(SAVE_ROOT / "rsg_tf_direction_dsi_a_tf.pdf", fig=fig, bbox_inches="tight")

### TF × RS slope and model-free TF shift in 8 direction bins

Model-free TF shift (`running_models/tf_shift.pickle`, `revisions/sftf/sftf_tf_shift.py`: fast − slow half of the
running frames, log2 TF) and TF × RS slope $a_{tf}$ (RS + TF shift model, cells with a significant rsg_tf fit)
against the preferred direction of the SF/TF fit (`fit_alpha0`, grating `Angle`; motion is towards angle + 180°),
for SF/TF-tuned direction-selective cells (`fit_dsi` > `DSI_THR`). Bins of 45° centred on 0°, 45°, …, 315°.
Black: bin median ± bootstrap 95% CI over cells; grey numbers: cells per bin; values clipped for display.
Kruskal-Wallis across bins, p from permuting bin labels within session.

In [ ]:
# TF x RS slope and model-free TF shift against the preferred direction of the SF/TF fit, in 8 bins of
# 45 deg centred on 0, 45, ..., 315 (grating Angle); direction-selective cells (fit_dsi > DSI_THR)
DIR_CENTRES = np.arange(0, 360, 45)
dir_rng = np.random.default_rng(0)
tf_shift = pd.read_pickle(SAVE_ROOT / "running_models" / "tf_shift.pickle")
dir_df = cells.loc[
    cells.sftf_tuned & (cells.fit_dsi > DSI_THR),
    ["session", "roi", "depth_tuned", "fit_alpha0"],
].assign(direction=lambda d: np.rad2deg(d.fit_alpha0) % 360)
dir_df["dir_bin"] = (np.round(dir_df.direction / 45).astype(int) % 8) * 45
dir_df = dir_df.merge(
    tf_shift[["session", "roi", "shift"]], on=["session", "roi"], how="left"
).merge(
    rs_fits[["session", "roi", "rsg_tf_a_tf", "rsg_tf_test_rsq_sig"]],
    on=["session", "roi"],
    how="left",
)
dir_df["depth_tuned"] = dir_df.depth_tuned.astype(bool)


def kruskal_perm(d, y, n=N_PERM):
    # Kruskal-Wallis H across direction bins; p with bin labels permuted within session
    d = d.dropna(subset=[y])
    groups = [d[y].to_numpy()[d.dir_bin.to_numpy() == c] for c in DIR_CENTRES]
    h = stats.kruskal(*[g for g in groups if len(g)]).statistic
    ranks, bins = stats.rankdata(d[y]), d.dir_bin.to_numpy()
    sessions = [np.flatnonzero(d.session.to_numpy() == s) for s in d.session.unique()]

    def h_stat(b):
        # H from ranks (no tie correction, same for observed and null)
        N = len(ranks)
        return 12 / (N * (N + 1)) * sum(
            ranks[b == c].sum() ** 2 / max((b == c).sum(), 1) for c in DIR_CENTRES
        ) - 3 * (N + 1)

    obs = h_stat(bins)
    null = np.empty(n)
    for i in range(n):
        b = bins.copy()
        for idx in sessions:
            b[idx] = dir_rng.permutation(b[idx])
        null[i] = h_stat(b)
    return h, (1 + np.sum(null >= obs)) / (n + 1), len(d)


def bin_medians(d, y, n_boots=2000):
    # median, bootstrap 95% CI over cells and n per bin
    out = []
    for c in DIR_CENTRES:
        v = d.loc[d.dir_bin == c, y].dropna().to_numpy()
        if len(v) > 2:
            boots = np.median(dir_rng.choice(v, (n_boots, len(v))), axis=1)
            out.append((np.median(v), *np.percentile(boots, [2.5, 97.5]), len(v)))
        else:
            out.append((np.median(v) if len(v) else np.nan, np.nan, np.nan, len(v)))
    return np.array(out)


DIR_PANELS = [
    (
        "shift",
        "Model-free TF shift\n(fast − slow, log2 TF)",
        2.5,
        lambda d: d["shift"].notna(),
    ),
    (
        "rsg_tf_a_tf",
        "TF × RS slope $a_{tf}$",
        1.5,
        lambda d: d.rsg_tf_test_rsq_sig.fillna(False),
    ),
]
DIR_PANEL_TITLES = ["model-free shift", "RS + TF shift model, rsg_tf sig."]
fig, axes = plt.subplots(2, 2, figsize=(17 * CM, 10 * CM))
fig.subplots_adjust(hspace=0.75, wspace=0.35)
for i, (group, sel_group) in enumerate(
    [("depth sel.", dir_df.depth_tuned), ("not depth sel.", ~dir_df.depth_tuned)]
):
    for j, ((y, ylabel, lim, sel), ptitle) in enumerate(
        zip(DIR_PANELS, DIR_PANEL_TITLES)
    ):
        ax = axes[i, j]
        d = dir_df[sel_group & sel(dir_df)]
        for c in DIR_CENTRES[::2]:
            ax.axvspan(c - 22.5, c + 22.5, color="0.94", lw=0, zorder=0)
        x = np.where(d.direction >= 337.5, d.direction - 360, d.direction)
        ax.scatter(x, d[y].clip(-lim, lim), s=4, c="#2a78d6", alpha=0.5, lw=0)
        bs = bin_medians(d, y)
        ax.errorbar(
            DIR_CENTRES,
            bs[:, 0],
            yerr=[bs[:, 0] - bs[:, 1], bs[:, 2] - bs[:, 0]],
            fmt="o",
            color="k",
            ms=3,
            capsize=1.5,
            lw=0.8,
        )
        for c, n_bin in zip(DIR_CENTRES, bs[:, 3]):
            ax.text(
                c,
                -lim * 1.12,
                f"{int(n_bin)}",
                ha="center",
                va="top",
                fontsize=5,
                color="0.4",
            )
        ax.axhline(0, color="0.5", lw=0.5, ls="--")
        ax.set_xticks(DIR_CENTRES)
        ax.set_xlim(-22.5, 337.5)
        ax.set_ylim(-lim * 1.2, lim)
        ax.set_xlabel("Preferred direction, SF/TF fit (°, grating Angle)")
        ax.set_ylabel(ylabel)
        h, p, n_cells = kruskal_perm(d, y)
        ax.set_title(
            f"{'AB'[j]}{i + 1}  {group}, {ptitle}\n"
            f"n = {n_cells}, Kruskal-Wallis H = {h:.1f}, p (perm. in session) = {p:.2g}",
            fontsize=6,
            loc="left",
        )
        ax.spines[["top", "right"]].set_visible(False)
style.savefig(SAVE_ROOT / "tf_shift_direction_bins.pdf", fig=fig, bbox_inches="tight")

### TF × RS slope against the shape of the RS/OF tuning (as figure 3K)

Polar plot of figure 3K (`figure_utils.rsof_integration.plot_angle_eccentricity_polar`): orientation of the RS/OF
2D Gaussian (`rsof_popt_closedloop_g2d`) against log2 elongation (σ_major / σ_minor), coloured by the TF × RS slope
(`rsg_tf_a_tf`, RS + TF shift model). Unlike figure 3K (treadmill, trial-average fits), the RS/OF fits here are the
per-frame free-locomotion fits of these sessions. Depth-selective cells only, with an RS/OF fit above its
empirical-null threshold, among the cells of the running-speed fits; left all of them, right those also
significant in the RS + TF shift model. Black edges show the points whose slope is near 0 (white).
Grey: elongation ≤ 1.4 (orientation undefined, as in figure 3K).

In [ ]:
from cottage_analysis.analysis import fit_gaussian_blob as fit_gb
from v1_depth_map.figure_utils.rsof_integration import plot_angle_eccentricity_polar

ELONGATION_CUTOFF = 1.4  # as figure 3K
SLOPE_LIM = 1.0  # colour scale of the TF x RS slope, +- this


def g2d_shape(session):
    popt = session_neurons(session).rsof_popt_closedloop_g2d
    return pd.DataFrame(
        dict(
            session=session,
            roi=np.arange(len(popt)),
            g2d_theta=popt.apply(fit_gb.get_gaussian_angle).values,
            g2d_elongation=(
                popt.apply(fit_gb.get_semimajor_length)
                / popt.apply(fit_gb.get_semiminor_length)
            ).values,
        )
    )


shape = pd.concat([g2d_shape(s) for s in SESSIONS], ignore_index=True)
polar_df = rs_fits.merge(shape, on=["session", "roi"], how="left").merge(
    cells[["session", "roi", "rsof_tuned"]], on=["session", "roi"], how="left"
)
polar_depth = polar_df[polar_df.depth_tuned.astype(bool)]
polar_sets = {
    "RS/OF fit sig.": polar_depth[polar_depth.rsof_tuned],
    "RS/OF and RS + TF shift fits sig.": polar_depth[
        polar_depth.rsof_tuned & polar_depth.rsg_tf_test_rsq_sig
    ],
}
fig, axes = plt.subplots(
    1, 2, figsize=(16 * CM, 8 * CM), subplot_kw=dict(projection="polar")
)
fig.subplots_adjust(wspace=0.45)
for ax, (name, d) in zip(axes, polar_sets.items()):
    elong_ok = d.g2d_elongation > ELONGATION_CUTOFF
    sc = plot_angle_eccentricity_polar(
        ax,
        d.loc[elong_ok, "g2d_theta"],
        None,
        FONTSIZE_DICT_SMALL,
        radial_scale="elongation",
        axis_ratio=d.loc[elong_ok, "g2d_elongation"],
        scale=0.5,
        rasterize_schematics="each",
        c=d.loc[elong_ok, "rsg_tf_a_tf"],
        cmap="RdBu_r",
        vmin=-SLOPE_LIM,
        vmax=SLOPE_LIM,
        s=12,
        alpha=0.9,
        edgecolors="k",
        linewidths=0.3,
    )
    ax.scatter(
        np.radians(d.loc[~elong_ok, "g2d_theta"]),
        np.clip(np.log2(d.loc[~elong_ok, "g2d_elongation"]), 0, None),
        c="#c3c2b7",
        s=8,
        linewidths=0,
        zorder=2,
    )
    rho, p = stats.spearmanr(
        np.abs(d.loc[elong_ok, "g2d_theta"] - 45), d.loc[elong_ok, "rsg_tf_a_tf"]
    )
    ax.set_title(
        f"{name}, n = {elong_ok.sum()} (+{(~elong_ok).sum()} grey)\n"
        f"|θ - 45°| vs slope: ρ = {rho:.2f}, p = {p:.2g}",
        fontsize=6,
        pad=14,
    )
cbar = fig.colorbar(sc, ax=axes, shrink=0.6, pad=0.08)
cbar.set_label("TF × RS slope (log TF per log RS)", fontsize=6)
cbar.ax.tick_params(labelsize=5)
style.savefig(SAVE_ROOT / "polar_tf_slope.pdf", fig=fig, bbox_inches="tight")